In [1]:
import pandas as pd

df = pd.read_csv("processed_data.csv")
print("Shape:", df.shape)
print(df["isFraud"].value_counts())
df.head()

Shape: (2770409, 7)
isFraud
0    2762196
1       8213
Name: count, dtype: int64


,step,amount,isFraud,type_TRANSFER,hour,day,log_amount
0,1,181.00,1,1,1,0,5.204007
1,1,181.00,1,0,1,0,5.204007
2,1,229133.94,0,0,1,0,12.342066
3,1,215310.30,0,1,1,0,12.279840
4,1,311685.89,0,1,1,0,12.649754


In [8]:
"""
Model stage - Financial Transaction Fraud Detection (PaySim).

What this script does:
1. Loads the cleaned data
2. Splits it by TIME (step): train 60% / validation 20% / test 20%
3. Trains 3 models: Logistic Regression, Random Forest, XGBoost
4. Picks the BEST model automatically by PR-AUC on the validation set
5. Picks the alert threshold on the validation set
6. Reports final results on the untouched test set
7. Saves: best_model.joblib, model_results.csv, pr_curves.png

Class imbalance is handled with class weights (no SMOTE), and any scaling
is fitted on the training set only, so there is no data leakage.
"""

import os
import time

import joblib
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

try:
    from xgboost import XGBClassifier

    HAS_XGB = True
except ImportError:
    HAS_XGB = False

# ----------------------------- Settings ------------------------------
RAW_FILE = "PS_20174392719_1491204439457_log.csv"  # original PaySim file
PROCESSED_FILE = "processed_data.csv"  # the team's processed data
TARGET = "isFraud"
TIME_COL = "step"
RANDOM_STATE = 42
# With a time-based split the test period has step/day values the model never
# saw in training, so they are used only for splitting, not as model inputs.
NOT_FEATURES = ["step", "day"]


def build_team_data_from_raw():
    """Rebuild the team's processed data (same columns as the team's notebook)
    from the original PaySim file: only TRANSFER and CASH_OUT rows, plus
    type_TRANSFER, hour, day and log_amount."""
    raw = pd.read_csv(RAW_FILE)
    df = raw[raw["type"].isin(["TRANSFER", "CASH_OUT"])].copy()
    df["type_TRANSFER"] = (df["type"] == "TRANSFER").astype(int)
    df["hour"] = df["step"] % 24
    df["day"] = df["step"] // 24
    df["log_amount"] = np.log1p(df["amount"])
    df = df[["step", "amount", "isFraud", "type_TRANSFER", "hour", "day", "log_amount"]]
    df.to_csv(PROCESSED_FILE, index=False)
    print("Built", PROCESSED_FILE, "from the raw file:", df.shape,
          "(the team's notebook reports 2770409 x 7)")


def choose_input():
    if os.path.exists(PROCESSED_FILE):
        return PROCESSED_FILE
    if os.path.exists(RAW_FILE):
        build_team_data_from_raw()
        return PROCESSED_FILE
    return "cleaned_fraud_data.csv"


def load_data(path):
    return pd.read_csv(path)


def prepare_like_team(df):
    """Match the team's notebook: keep only TRANSFER and CASH_OUT transactions
    (fraud exists only in these two types), keep type_TRANSFER as a flag."""
    if "type_TRANSFER" in df.columns and "type_CASH_OUT" in df.columns:
        df = df[(df["type_TRANSFER"] == 1) | (df["type_CASH_OUT"] == 1)].copy()
        drop = [c for c in ["type_CASH_IN", "type_CASH_OUT", "type_DEBIT", "type_PAYMENT"]
                if c in df.columns]
        df = df.drop(columns=drop)
        print("Kept only TRANSFER / CASH_OUT rows:", len(df))
    return df


def add_features(df):
    """Time features, log amount, and balance-mismatch features.
    Each one is added only if the needed columns exist and it is not there yet."""
    if "hour" not in df.columns and TIME_COL in df.columns:
        df["hour"] = df[TIME_COL] % 24
        df["day"] = df[TIME_COL] // 24
    if "log_amount" not in df.columns and "amount" in df.columns:
        df["log_amount"] = np.log1p(df["amount"])
    needed = ["amount", "oldbalanceOrg", "newbalanceOrig",
              "oldbalanceDest", "newbalanceDest"]
    if all(c in df.columns for c in needed) and "errorBalanceOrig" not in df.columns:
        df["errorBalanceOrig"] = df["newbalanceOrig"] + df["amount"] - df["oldbalanceOrg"]
        df["errorBalanceDest"] = df["oldbalanceDest"] + df["amount"] - df["newbalanceDest"]
    return df


def time_split(df):
    """Train 60% / validation 20% / test 20% by time (never random)."""
    df = df.sort_values(TIME_COL).reset_index(drop=True)
    n = len(df)
    train = df.iloc[: int(n * 0.6)]
    val = df.iloc[int(n * 0.6): int(n * 0.8)]
    test = df.iloc[int(n * 0.8):]
    return train, val, test


def xy(part):
    drop = [TARGET] + [c for c in NOT_FEATURES if c in part.columns]
    return part.drop(columns=drop), part[TARGET]


def build_models(y_train):
    ratio = (y_train == 0).sum() / max((y_train == 1).sum(), 1)
    models = {
        "Logistic Regression": make_pipeline(
            StandardScaler(),
            LogisticRegression(class_weight="balanced", max_iter=1000),
        ),
        "Random Forest": RandomForestClassifier(
            n_estimators=100,
            max_depth=14,
            min_samples_leaf=5,
            max_samples=0.3,
            class_weight="balanced_subsample",
            n_jobs=-1,
            random_state=RANDOM_STATE,
        ),
    }
    if HAS_XGB:
        models["XGBoost"] = XGBClassifier(
            n_estimators=300,
            max_depth=6,
            learning_rate=0.1,
            scale_pos_weight=ratio,
            tree_method="hist",
            eval_metric="aucpr",
            n_jobs=-1,
            random_state=RANDOM_STATE,
        )
    else:
        print("xgboost is not installed -> using HistGradientBoosting instead "
              "(run: pip install xgboost)")
        models["Gradient Boosting (XGBoost substitute)"] = HistGradientBoostingClassifier(
            class_weight="balanced", random_state=RANDOM_STATE
        )
    return models


def best_threshold(y_true, prob):
    """Threshold with the best F1 on the validation set."""
    p, r, t = precision_recall_curve(y_true, prob)
    f1 = 2 * p[:-1] * r[:-1] / (p[:-1] + r[:-1] + 1e-12)
    return float(t[np.argmax(f1)])


def evaluate(y_true, prob, threshold):
    pred = (prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    return {
        "Precision": precision_score(y_true, pred, zero_division=0),
        "Recall": recall_score(y_true, pred, zero_division=0),
        "F1": f1_score(y_true, pred, zero_division=0),
        "PR-AUC": average_precision_score(y_true, prob),
        "TP": tp, "FP": fp, "FN": fn, "TN": tn,
    }


def main():
    input_file = choose_input()
    print("Using data file:", input_file)
    df = load_data(input_file)
    df = prepare_like_team(df)
    df = add_features(df)
    print("Data shape:", df.shape)
    print("Fraud ratio: {:.4%}".format(df[TARGET].mean()))

    train, val, test = time_split(df)
    X_train, y_train = xy(train)
    X_val, y_val = xy(val)
    X_test, y_test = xy(test)
    print("Model inputs:", list(X_train.columns))
    print(f"Train {len(train):,} | Validation {len(val):,} | Test {len(test):,}")
    print("Fraud cases -> train {}, val {}, test {}".format(
        y_train.sum(), y_val.sum(), y_test.sum()))

    models = build_models(y_train)
    fitted, val_prob, rows = {}, {}, []

    for name, model in models.items():
        print(f"\nTraining {name} ...")
        start = time.time()
        model.fit(X_train, y_train)
        prob = model.predict_proba(X_val)[:, 1]
        fitted[name], val_prob[name] = model, prob
        rows.append({
            "Model": name,
            "Val PR-AUC": average_precision_score(y_val, prob),
            "Train time (s)": round(time.time() - start, 1),
        })

    selection = pd.DataFrame(rows).sort_values("Val PR-AUC", ascending=False)
    print("\n=== Model comparison on validation set ===")
    print(selection.to_string(index=False))

    best_name = selection.iloc[0]["Model"]
    best_model = fitted[best_name]
    threshold = best_threshold(y_val, val_prob[best_name])
    print(f"\nBest model: {best_name}")
    print(f"Chosen threshold (best F1 on validation): {threshold:.4f}")

    # Final results on the untouched test set, for all models (same threshold rule)
    final_rows, curves = [], {}
    for name, model in fitted.items():
        thr = threshold if name == best_name else best_threshold(y_val, val_prob[name])
        prob_test = model.predict_proba(X_test)[:, 1]
        res = evaluate(y_test, prob_test, thr)
        res["Model"] = name
        res["Threshold"] = round(thr, 4)
        final_rows.append(res)
        curves[name] = precision_recall_curve(y_test, prob_test)

    results = pd.DataFrame(final_rows)[
        ["Model", "Precision", "Recall", "F1", "PR-AUC", "Threshold",
         "TP", "FP", "FN", "TN"]
    ].sort_values("PR-AUC", ascending=False)
    print("\n=== Final results on TEST set ===")
    print(results.round(4).to_string(index=False))
    results.to_csv("model_results.csv", index=False)

    # Precision-Recall curves
    plt.figure(figsize=(7, 5))
    for name, (p, r, _) in curves.items():
        plt.plot(r, p, label=name)
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title("Precision-Recall curves (test set)")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig("pr_curves.png", dpi=150)

    # Feature importance for tree models
    if hasattr(best_model, "feature_importances_"):
        imp = pd.Series(best_model.feature_importances_, index=X_train.columns)
        print("\nTop features:")
        print(imp.sort_values(ascending=False).head(10).round(4).to_string())

    # Save the best model with everything the dashboard needs
    joblib.dump(
        {"model": best_model, "threshold": threshold,
         "features": list(X_train.columns), "name": best_name},
        "best_model.joblib",
    )
    print("\nSaved: best_model.joblib, model_results.csv, pr_curves.png")


if __name__ == "__main__":
    main()

Matplotlib is building the font cache; this may take a moment.


Using data file: processed_data.csv
Data shape: (2770409, 7)
Fraud ratio: 0.2965%
Model inputs: ['amount', 'type_TRANSFER', 'hour', 'log_amount']
Train 1,662,245 | Validation 554,082 | Test 554,082
Fraud cases -> train 3179, val 774, test 4260
xgboost is not installed -> using HistGradientBoosting instead (run: pip install xgboost)

Training Logistic Regression ...

Training Random Forest ...

Training Gradient Boosting (XGBoost substitute) ...

=== Model comparison on validation set ===
                                 Model  Val PR-AUC  Train time (s)
Gradient Boosting (XGBoost substitute)    0.093587            18.9
                         Random Forest    0.071857           103.8
                   Logistic Regression    0.005369             3.3

Best model: Gradient Boosting (XGBoost substitute)
Chosen threshold (best F1 on validation): 0.9965

=== Final results on TEST set ===
                                 Model  Precision  Recall     F1  PR-AUC  Threshold  TP   FP   FN     T